# Credit Risk Modeling using Machine Learning

## Predicting Good vs. Bad Credit Risk

This project develops a machine learning model to predict credit risk
and classify customers as Good or Bad credit risk.

In [1]:
import pandas as pd

## 1. Data Loading

The German Credit dataset is loaded into a Pandas DataFrame for analysis and modeling.

In [2]:
df = pd.read_csv('/content/german_credit_data.csv')

## 2. Data Understanding

The dataset structure, dimensions, sample records, data types, missing values, and duplicate records are examined to understand the quality and structure of the data.

In [3]:
df.shape

(1000, 21)

In [4]:
df.head()

,laufkont,laufzeit,moral,verw,hoehe,sparkont,beszeit,rate,famges,buerge,...,verm,alter,weitkred,wohn,bishkred,beruf,pers,telef,gastarb,kredit
0,1,18,4,2,1049,1,2,4,2,1,...,2,21,3,1,1,3,2,1,2,1
1,1,9,4,0,2799,1,3,2,3,1,...,1,36,3,1,2,3,1,1,2,1
2,2,12,2,9,841,2,4,2,2,1,...,1,23,3,1,1,2,2,1,2,1
3,1,12,4,0,2122,1,3,3,3,1,...,1,39,3,1,2,2,1,1,1,1
4,1,12,4,0,2171,1,3,4,3,1,...,2,38,1,2,2,2,2,1,1,1


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 21 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   laufkont  1000 non-null   int64
 1   laufzeit  1000 non-null   int64
 2   moral     1000 non-null   int64
 3   verw      1000 non-null   int64
 4   hoehe     1000 non-null   int64
 5   sparkont  1000 non-null   int64
 6   beszeit   1000 non-null   int64
 7   rate      1000 non-null   int64
 8   famges    1000 non-null   int64
 9   buerge    1000 non-null   int64
 10  wohnzeit  1000 non-null   int64
 11  verm      1000 non-null   int64
 12  alter     1000 non-null   int64
 13  weitkred  1000 non-null   int64
 14  wohn      1000 non-null   int64
 15  bishkred  1000 non-null   int64
 16  beruf     1000 non-null   int64
 17  pers      1000 non-null   int64
 18  telef     1000 non-null   int64
 19  gastarb   1000 non-null   int64
 20  kredit    1000 non-null   int64
dtypes: int64(21)
memory usage: 164.2 KB


In [6]:
print(df.isnull().sum())

laufkont    0
laufzeit    0
moral       0
verw        0
hoehe       0
sparkont    0
beszeit     0
rate        0
famges      0
buerge      0
wohnzeit    0
verm        0
alter       0
weitkred    0
wohn        0
bishkred    0
beruf       0
pers        0
telef       0
gastarb     0
kredit      0
dtype: int64


In [7]:
print(df.duplicated().sum())

0


## 3. Target Variable Analysis

The target variable is `credit_risk`, where:

- 0 = Bad Credit Risk
- 1 = Good Credit Risk

The distribution of the target variable is examined to understand the balance between Good and Bad customers.

In [8]:
df['kredit'].value_counts()

,count
kredit,
1,700
0,300


In [9]:
df = df.rename(columns={
    'laufkont': 'checking_account_status',
    'laufzeit': 'loan_duration_months',
    'moral': 'credit_history',
    'verw': 'loan_purpose',
    'hoehe': 'credit_amount',
    'sparkont': 'savings_account',
    'beszeit': 'employment_duration',
    'rate': 'installment_rate',
    'famges': 'personal_status_sex',
    'buerge': 'other_debtors',
    'wohnzeit': 'residence_duration',
    'verm': 'property',
    'alter': 'age',
    'weitkred': 'other_installment_plans',
    'wohn': 'housing',
    'bishkred': 'number_of_credits',
    'beruf': 'job',
    'pers': 'people_liable',
    'telef': 'telephone',
    'gastarb': 'foreign_worker',
    'kredit': 'credit_risk'
})

In [10]:
df.head()

,checking_account_status,loan_duration_months,credit_history,loan_purpose,credit_amount,savings_account,employment_duration,installment_rate,personal_status_sex,other_debtors,...,property,age,other_installment_plans,housing,number_of_credits,job,people_liable,telephone,foreign_worker,credit_risk
0,1,18,4,2,1049,1,2,4,2,1,...,2,21,3,1,1,3,2,1,2,1
1,1,9,4,0,2799,1,3,2,3,1,...,1,36,3,1,2,3,1,1,2,1
2,2,12,2,9,841,2,4,2,2,1,...,1,23,3,1,1,2,2,1,2,1
3,1,12,4,0,2122,1,3,3,3,1,...,1,39,3,1,2,2,1,1,1,1
4,1,12,4,0,2171,1,3,4,3,1,...,2,38,1,2,2,2,2,1,1,1


## 4. Feature Exploration

The number of unique values in each feature is examined to understand the nature of the variables and determine appropriate preprocessing techniques.

In [11]:
df.nunique().sort_values()

,0
credit_risk,2
foreign_worker,2
telephone,2
people_liable,2
other_installment_plans,3
housing,3
other_debtors,3
checking_account_status,4
number_of_credits,4
installment_rate,4


## 5. Feature and Target Definition

The dataset is separated into:

- Features (X): variables used to predict credit risk.
- Target (y): `credit_risk`.

In [12]:
x = df.drop('credit_risk',axis=1)
y = df['credit_risk']

## 6. Feature Classification

The input features are divided into three groups based on their data type and meaning:

- Numerical Features
- Ordinal Features
- Categorical Features

This classification determines the appropriate preprocessing method for each group.

In [13]:
numerical_features = [
    'loan_duration_months',
    'credit_amount',
    'age'
]
ordinal_features = [
    'checking_account_status',
    'credit_history',
    'savings_account',
    'employment_duration',
    'installment_rate',
    'residence_duration',
    'number_of_credits',
    'job'
]
categorical_features = [
    'loan_purpose',
    'personal_status_sex',
    'other_debtors',
    'property',
    'other_installment_plans',
    'housing',
    'telephone',
    'foreign_worker'
]

## 7. Train/Test Split

The dataset is divided into training and test sets.

The training set is used to train the models, while the test set is kept separate for final performance evaluation.

In [14]:
from sklearn.model_selection import train_test_split

target = 'credit_risk'
x = df.drop(columns=[target])
y = df[target]

x_train,x_test,y_train,y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [15]:
print('x_train:',x_train.shape)
print('x_test:',x_test.shape)
print('y_train:',y_train.shape)
print('y_test:',y_test.shape)

x_train: (800, 20)
x_test: (200, 20)
y_train: (800,)
y_test: (200,)


## 8. Data Preprocessing

Different preprocessing techniques are applied according to the feature type:

- Numerical features → Standard Scaling
- Ordinal features → Ordinal Encoding
- Categorical features → One-Hot Encoding

A ColumnTransformer is used to combine all preprocessing steps into a single preprocessing pipeline.

In [16]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),

        ('ord', OrdinalEncoder(
            handle_unknown='use_encoded_value',
            unknown_value=-1
        ), ordinal_features),

        ('cat', OneHotEncoder(
            handle_unknown='ignore'
        ), categorical_features)
    ]
)

x_train_processed = preprocessor.fit_transform(x_train)

x_test_processed = preprocessor.transform(x_test)

## 9. Model Training

### Logistic Regression

Logistic Regression is used as the baseline classification model for predicting Good and Bad credit risk.

In [17]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression()
model.fit(x_train_processed,y_train)

LogisticRegression()

In [18]:
y_pred = model.predict(x_test_processed)

## 10. Model Evaluation

The model is evaluated using Accuracy, Precision, Recall, F1-score, Confusion Matrix, and ROC-AUC.

Special attention is given to Bad Recall because identifying high-risk customers is important in credit risk assessment.

In [19]:
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(y_test,y_pred)
print("Accuracy:",accuracy)

Accuracy: 0.78


In [20]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

print(cm)

[[ 35  25]
 [ 19 121]]


In [21]:
print(y.value_counts())

credit_risk
1    700
0    300
Name: count, dtype: int64


In [22]:
print(df[['credit_risk']].head(10))

   credit_risk
0            1
1            1
2            1
3            1
4            1
5            1
6            1
7            1
8            1
9            1


In [23]:
from sklearn.metrics import precision_score , recall_score , f1_score

precision = precision_score(y_test,y_pred)
recall = recall_score(y_test,y_pred)
f1 = f1_score(y_test,y_pred)

print("precision : ", precision)
print("recall : " , recall)
print("f1_score : " , f1)


precision :  0.8287671232876712
recall :  0.8642857142857143
f1_score :  0.8461538461538461


In [24]:
precision_bad = precision_score(y_test, y_pred, pos_label=0)
recall_bad = recall_score(y_test, y_pred, pos_label=0)
f1_bad = f1_score(y_test, y_pred, pos_label=0)

print("Bad Precision:", precision_bad)
print("Bad Recall:", recall_bad)
print("Bad F1-score:", f1_bad)

Bad Precision: 0.6481481481481481
Bad Recall: 0.5833333333333334
Bad F1-score: 0.6140350877192983


In [25]:
from sklearn.metrics import roc_auc_score

y_proba = model.predict_proba(x_test_processed)[:, 1]

roc_auc = roc_auc_score(y_test, y_proba)

print("ROC-AUC:", roc_auc)

ROC-AUC: 0.8338095238095238


In [26]:
# Logistic Regression

# Accuracy       = 78.0%
# Good Precision = 82.9%
# Good Recall    = 86.4%
# Good F1        = 84.6%

# Bad Precision  = 64.8%
# Bad Recall     = 58.3%
# Bad F1         = 61.4%

# ROC-AUC        = 83.4%

## 11. Model Comparison

### Decision Tree

A Decision Tree is trained and evaluated as an alternative classification model.

In [27]:
from sklearn.tree import DecisionTreeClassifier

tree_model = DecisionTreeClassifier (random_state=42)
tree_model.fit(x_train_processed,y_train)

DecisionTreeClassifier(random_state=42)

In [28]:
y_pred_tree = tree_model.predict(x_test_processed)

In [29]:
accuracy_tree = accuracy_score(y_test, y_pred_tree)

precision_tree = precision_score(y_test, y_pred_tree, pos_label=0)

recall_tree = recall_score(y_test, y_pred_tree, pos_label=0)

f1_tree = f1_score(y_test, y_pred_tree, pos_label=0)

y_proba_tree = tree_model.predict_proba(x_test_processed)[:, 1]

roc_auc_tree = roc_auc_score(y_test, y_proba_tree)

print("Decision Tree Accuracy:", accuracy_tree)
print("Decision Tree Bad Precision:", precision_tree)
print("Decision Tree Bad Recall:", recall_tree)
print("Decision Tree Bad F1:", f1_tree)
print("Decision Tree ROC-AUC:", roc_auc_tree)

Decision Tree Accuracy: 0.68
Decision Tree Bad Precision: 0.46774193548387094
Decision Tree Bad Recall: 0.48333333333333334
Decision Tree Bad F1: 0.47540983606557374
Decision Tree ROC-AUC: 0.6238095238095237


In [30]:
# Decision Tree

# Accuracy       = 68.0%
# Good Precision = -
# Good Recall    = -
# Good F1        = -

# Bad Precision  = 46.8%
# Bad Recall     = 48.3%
# Bad F1         = 47.5%

# ROC-AUC        = 62.4%

### Random Forest

Random Forest is trained and evaluated as another alternative to Logistic Regression and Decision Tree.

In [31]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_model.fit(x_train_processed, y_train)

y_pred_rf = rf_model.predict(x_test_processed)

accuracy_rf = accuracy_score(y_test, y_pred_rf)

precision_rf = precision_score(y_test, y_pred_rf, pos_label=0)

recall_rf = recall_score(y_test, y_pred_rf, pos_label=0)

f1_rf = f1_score(y_test, y_pred_rf, pos_label=0)

y_proba_rf = rf_model.predict_proba(x_test_processed)[:, 1]

roc_auc_rf = roc_auc_score(y_test, y_proba_rf)

print("Random Forest Accuracy:", accuracy_rf)
print("Random Forest Bad Precision:", precision_rf)
print("Random Forest Bad Recall:", recall_rf)
print("Random Forest Bad F1:", f1_rf)
print("Random Forest ROC-AUC:", roc_auc_rf)

Random Forest Accuracy: 0.755
Random Forest Bad Precision: 0.6341463414634146
Random Forest Bad Recall: 0.43333333333333335
Random Forest Bad F1: 0.5148514851485149
Random Forest ROC-AUC: 0.817202380952381


In [32]:
# Random Forest

# Accuracy       = 75.5%
# Good Precision = -
# Good Recall    = -
# Good F1        = -

# Bad Precision  = 63.4%
# Bad Recall     = 43.3%
# Bad F1         = 51.5%

# ROC-AUC        = 81.7%

## 12. Model Comparison

The performance of the three models is compared using Accuracy, Bad Precision, Bad Recall, Bad F1-score, and ROC-AUC.

The comparison helps identify the most suitable model for the credit risk classification problem.

In [33]:
#"Logistic Regression performed best among the three models. It achieved the highest accuracy, Bad-class recall, F1-score, and ROC-AUC. Since this is a credit risk problem, I paid particular attention to Bad-class recall because missing a bad customer can represent a significant credit risk."

In [34]:
threshold = 0.6

y_pred_threshold = (y_proba >= threshold ).astype(int)

recall_bad_threshold = recall_score(
    y_test,
    y_pred_threshold,
    pos_label=0
)

print("Threshold:", threshold)
print("Bad Recall:", recall_bad_threshold)

Threshold: 0.6
Bad Recall: 0.6166666666666667


## 13. Threshold Optimization

The classification threshold is adjusted to improve the model's ability to identify Bad customers.

Different thresholds are evaluated based on Bad Recall, Precision, F1-score, and Accuracy.

In [35]:
import numpy as np

thresholds = np.arange(0.1, 1.0, 0.01)

results = []

for threshold in thresholds:
    y_pred_threshold = (y_proba >= threshold).astype(int)

    bad_recall = recall_score(
        y_test,
        y_pred_threshold,
        pos_label=0
    )

    results.append((threshold, bad_recall))

best_threshold, best_bad_recall = max(
    results,
    key=lambda x: x[1]
)

print("Best Threshold:", best_threshold)
print("Best Bad Recall:", best_bad_recall)

Best Threshold: 0.9499999999999995
Best Bad Recall: 1.0


In [36]:
thresholds = np.arange(0.1, 1.0, 0.01)

threshold_results = []

for threshold in thresholds:

    y_pred_threshold = (y_proba >= threshold).astype(int)

    accuracy = accuracy_score(y_test, y_pred_threshold)

    bad_precision = precision_score(
        y_test,
        y_pred_threshold,
        pos_label=0,
        zero_division=0
    )

    bad_recall = recall_score(
        y_test,
        y_pred_threshold,
        pos_label=0,
        zero_division=0
    )

    bad_f1 = f1_score(
        y_test,
        y_pred_threshold,
        pos_label=0,
        zero_division=0
    )

    threshold_results.append(
        [threshold, accuracy, bad_precision, bad_recall, bad_f1]
    )

### Optimal Threshold Selection

The threshold that provides the best balance for identifying Bad customers is selected based on the evaluation results.

In [37]:
import pandas as pd

threshold_df = pd.DataFrame(
    threshold_results,
    columns=[
        'Threshold',
        'Accuracy',
        'Bad Precision',
        'Bad Recall',
        'Bad F1'
    ]
)

threshold_df

,Threshold,Accuracy,Bad Precision,Bad Recall,Bad F1
0,0.10,0.705,0.666667,0.033333,0.063492
1,0.11,0.715,0.800000,0.066667,0.123077
2,0.12,0.715,0.800000,0.066667,0.123077
3,0.13,0.720,0.833333,0.083333,0.151515
4,0.14,0.725,0.857143,0.100000,0.179104
...,...,...,...,...,...
85,0.95,0.415,0.338983,1.000000,0.506329
86,0.96,0.375,0.324324,1.000000,0.489796
87,0.97,0.355,0.317460,1.000000,0.481928
88,0.98,0.330,0.309278,1.000000,0.472441


In [38]:
best_row = threshold_df.loc[threshold_df['Bad F1'].idxmax()]

print("Best Threshold:", best_row['Threshold'])
print("Accuracy:", best_row['Accuracy'])
print("Bad Precision:", best_row['Bad Precision'])
print("Bad Recall:", best_row['Bad Recall'])
print("Bad F1:", best_row['Bad F1'])

Best Threshold: 0.7099999999999996
Accuracy: 0.765
Bad Precision: 0.5783132530120482
Bad Recall: 0.8
Bad F1: 0.6713286713286714


In [39]:
#1️⃣1️⃣ Model Selection

final_model = model
final_threshold = 0.71

print("Selected Model: Logistic Regression")
print("Selected Threshold:", final_threshold)

Selected Model: Logistic Regression
Selected Threshold: 0.71


In [40]:
#1️⃣2️⃣ Feature Importance / Explainability

feature_names = preprocessor.get_feature_names_out()

print(len(feature_names))
print(feature_names)

42
['num__loan_duration_months' 'num__credit_amount' 'num__age'
 'ord__checking_account_status' 'ord__credit_history'
 'ord__savings_account' 'ord__employment_duration' 'ord__installment_rate'
 'ord__residence_duration' 'ord__number_of_credits' 'ord__job'
 'cat__loan_purpose_0' 'cat__loan_purpose_1' 'cat__loan_purpose_2'
 'cat__loan_purpose_3' 'cat__loan_purpose_4' 'cat__loan_purpose_5'
 'cat__loan_purpose_6' 'cat__loan_purpose_8' 'cat__loan_purpose_9'
 'cat__loan_purpose_10' 'cat__personal_status_sex_1'
 'cat__personal_status_sex_2' 'cat__personal_status_sex_3'
 'cat__personal_status_sex_4' 'cat__other_debtors_1'
 'cat__other_debtors_2' 'cat__other_debtors_3' 'cat__property_1'
 'cat__property_2' 'cat__property_3' 'cat__property_4'
 'cat__other_installment_plans_1' 'cat__other_installment_plans_2'
 'cat__other_installment_plans_3' 'cat__housing_1' 'cat__housing_2'
 'cat__housing_3' 'cat__telephone_1' 'cat__telephone_2'
 'cat__foreign_worker_1' 'cat__foreign_worker_2']


### Model Coefficients

The model coefficients are extracted and organized into a table to examine the direction and magnitude of each feature's contribution.

In [41]:
coefficients = final_model.coef_[0]

feature_importance = pd.DataFrame({
    'Feature' : feature_names ,
    'coefficient' : coefficients
})

feature_importance

,Feature,coefficient
0,num__loan_duration_months,-0.331981
1,num__credit_amount,-0.256863
2,num__age,0.041456
3,ord__checking_account_status,0.532158
4,ord__credit_history,0.370775
5,ord__savings_account,0.279943
6,ord__employment_duration,0.092802
7,ord__installment_rate,-0.308960
8,ord__residence_duration,0.037857
9,ord__number_of_credits,-0.201469


## 16. Hyperparameter Tuning

GridSearchCV is used to test different Logistic Regression hyperparameter combinations and identify the configuration that provides the best cross-validated ROC-AUC.

In [42]:
#1️⃣3️⃣ Hyperparameter Tuning

from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression

logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

param_grid = {
    'C': [0.01, 0.1, 1, 10, 100],
    'solver': ['liblinear', 'lbfgs']
}

grid_search = GridSearchCV(
    estimator=logistic_model,
    param_grid=param_grid,
    scoring='roc_auc',
    cv=5,
    n_jobs=-1
)

grid_search.fit(x_train_processed, y_train)

best_model = grid_search.best_estimator_

print("Best Parameters:", grid_search.best_params_)
print("Best CV ROC-AUC:", grid_search.best_score_)

Best Parameters: {'C': 0.1, 'solver': 'lbfgs'}
Best CV ROC-AUC: 0.7789806547619047


## 17. Final Model Evaluation

The tuned Logistic Regression model is evaluated on the unseen test set to obtain the final model performance.

In [43]:
#1️⃣4️⃣ Final Model

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# Final Model Predictions
y_pred_final = best_model.predict(x_test_processed)
y_proba_final = best_model.predict_proba(x_test_processed)[:, 1]

# Final Model Evaluation
accuracy_final = accuracy_score(y_test, y_pred_final)

precision_bad_final = precision_score(
    y_test, y_pred_final, pos_label=0, zero_division=0
)

recall_bad_final = recall_score(
    y_test, y_pred_final, pos_label=0, zero_division=0
)

f1_bad_final = f1_score(
    y_test, y_pred_final, pos_label=0, zero_division=0
)

roc_auc_final = roc_auc_score(y_test, y_proba_final)

print("Final Model Accuracy:", accuracy_final)
print("Final Model Bad Precision:", precision_bad_final)
print("Final Model Bad Recall:", recall_bad_final)
print("Final Model Bad F1:", f1_bad_final)
print("Final Model ROC-AUC:", roc_auc_final)

Final Model Accuracy: 0.795
Final Model Bad Precision: 0.6938775510204082
Final Model Bad Recall: 0.5666666666666667
Final Model Bad F1: 0.6238532110091743
Final Model ROC-AUC: 0.8360714285714286


## 18. Final Risk Threshold

The optimized threshold of 0.71 is applied to the tuned model to improve the detection of Bad customers.

In [44]:
final_threshold = 0.71

y_pred_final_threshold = (
    y_proba_final >= final_threshold
).astype(int)

print("Final Threshold:", final_threshold)

print(
    "Final Bad Recall:",
    recall_score(
        y_test,
        y_pred_final_threshold,
        pos_label=0,
        zero_division=0
    )
)

print(
    "Final Bad Precision:",
    precision_score(
        y_test,
        y_pred_final_threshold,
        pos_label=0,
        zero_division=0
    )
)

print(
    "Final Bad F1:",
    f1_score(
        y_test,
        y_pred_final_threshold,
        pos_label=0,
        zero_division=0
    )
)

Final Threshold: 0.71
Final Bad Recall: 0.8
Final Bad Precision: 0.5647058823529412
Final Bad F1: 0.6620689655172414


## 19. Credit Risk Interpretation

The final model predictions are converted into Good and Bad credit risk classifications using the optimized threshold of 0.71.

- 0 = Bad Credit Risk
- 1 = Good Credit Risk

The optimized threshold prioritizes the detection of Bad customers, which is important from a credit risk perspective.

In [45]:
#1️⃣5️⃣ Credit Risk Interpretation

final_prediction = (y_proba_final >= 0.71).astype(int)

print(final_prediction[:20])

[0 1 1 1 0 1 0 0 1 1 0 0 1 0 0 1 1 1 1 1]


# 20. Conclusion

This project demonstrates an end-to-end machine learning approach to credit risk classification.

The workflow included data preprocessing, model training, model comparison, threshold optimization, feature importance analysis, hyperparameter tuning, and final model evaluation.

The final Logistic Regression model achieved a ROC-AUC of 83.6%, while the optimized threshold of 0.71 improved Bad customer recall to 80%.

This demonstrates how machine learning can support credit risk assessment by identifying potentially high-risk customers.